# Task 2 — Validation of Temporal Knowledge Graphs (Google Colab / Jupyter)

Этот блокнот является **естественным продолжением Task 1**.

Он сначала ищет **локальный исправленный репозиторий рядом с собой** и только если не находит его — использует `git clone` официального GitHub-репозитория.

Эксперт загружает YAML-файл, созданный в первом блокноте, после чего блокнот автоматически:
- строит **эталонный (gold) граф**, который точно воспроизводит ручную reasoning-схему из YAML;
- при желании строит **автоматический temporal KG** по тем же статьям через пайплайн репозитория;
- сохраняет обе версии графа, таблицы триплетов, comparison summary и визуализации;
- показывает эксперту **текстовое представление триплетов** и **интерактивные визуализации** для ручной валидации.

Блокнот **не патчит файлы репозитория**. Он только ставит зависимости, импортирует уже готовый API репозитория и запускает workflow.


# Пошаговый туториал

## Шаг 1. Подготовьте репозиторий
Рекомендуемый вариант — распаковать архив `top-papers-graph-fixed-v5.zip` в `/content` или рядом с ноутбуком.

## Шаг 2. Запустите ячейку «Установка и импорт»
Она:
- найдёт локальный репозиторий автоматически;
- только если локальный репозиторий не найден, клонирует официальный GitHub-репозиторий;
- установит зависимости для Task 2 notebook;
- добавит локальный `src` в `sys.path` как надёжный fallback;
- импортирует функции Task 2.

## Шаг 3. Запустите ячейку «Вспомогательные функции»
Она подготовит UI, рендеринг таблиц и графов.

## Шаг 4. Запустите ячейку «Форма Task 2»
В ней нужно:
1. загрузить YAML из Task 1;
2. выбрать, строить ли автоматический граф;
3. нажать кнопку запуска.


In [ ]:
# @title
# Установка и импорт (запустите эту ячейку)
import json, os, sys, tempfile, subprocess
from pathlib import Path


def run(cmd, cwd=None):
    print('>', ' '.join(map(str, cmd)))
    subprocess.check_call(cmd, cwd=cwd)


def find_repo_root() -> Path | None:
    candidates = []
    env_dir = os.environ.get('TPG_REPO_DIR')
    if env_dir:
        candidates.append(Path(env_dir))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent, Path('/content'), Path('/mnt/data')])
    search_bases = [Path('/content'), Path('/mnt/data'), cwd, cwd.parent]
    patterns = (
        'top-papers-graph-fixed-v5',
        'top-papers-graph-fixed-v4',
        'top-papers-graph-fixed-v3',
        'top-papers-graph-fixed*',
        'top-papers-graph-main',
        'top-papers-graph',
    )
    for base in search_bases:
        if not base.exists():
            continue
        for pattern in patterns:
            candidates.extend(base.glob(pattern))
    seen = set()
    uniq = []
    for c in candidates:
        try:
            r = c.resolve()
        except Exception:
            r = c
        key = str(r)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(r)
    for c in uniq:
        if (c / 'pyproject.toml').exists() and (c / 'src' / 'scireason').exists():
            return c
    return None


REPO_DIR = find_repo_root()
REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'
if REPO_DIR is None:
    REPO_DIR = Path('/content/top-papers-graph') if Path('/content').exists() else Path.cwd() / 'top-papers-graph'
    if not REPO_DIR.exists():
        run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])

SRC_DIR = REPO_DIR / 'src'
if not SRC_DIR.exists():
    raise FileNotFoundError(f'Не найден каталог src в репозитории: {SRC_DIR}')

# 1) Лёгкий UI-стек для notebook. pandas здесь специально НЕ ставим отдельно,
#    чтобы не получить ABI-конфликт с numpy после editable-install репозитория.
run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'ipywidgets', 'pyyaml', 'requests',
    'panel', 'hvplot', 'holoviews', 'bokeh', 'pyvis', 'jupyter_bokeh'
])

# 2) Ставим extras репозитория.
#    Сначала пробуем единый extra из исправленного архива, затем fallback для GitHub upstream.
try:
    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '-e', f'{REPO_DIR}[task2_notebook]'])
except Exception:
    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '-e', f'{REPO_DIR}[temporal,mm]'])

# 3) В Colab часто остаётся предустановленный pandas, собранный под другой ABI NumPy.
#    Принудительно выравниваем совместимую пару после editable-install.
run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-cache-dir',
    'numpy>=1.26.4,<2', 'pandas>=2.2.2,<2.4'
])

# 4) Даже если editable-install не прописал import hook, ноутбук всё равно должен работать.
#    Поэтому всегда добавляем локальный src в sys.path как надёжный fallback.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# На случай, если kernel уже видел старые модули
for mod in list(sys.modules):
    if mod == 'numpy' or mod.startswith('numpy.') or mod == 'pandas' or mod.startswith('pandas.'):
        del sys.modules[mod]

import numpy as np
import pandas as pd
import ipywidgets as W
from IPython.display import display, Markdown, HTML, clear_output

try:
    import panel as pn
    pn.extension()
except Exception:
    pn = None

try:
    import hvplot  # noqa: F401
    import hvplot.networkx  # noqa: F401
except Exception:
    pass

try:
    from scireason.task2_validation import (
        build_task2_validation_bundle,
        load_task1_yaml,
        make_hvplot_payload,
    )
except Exception:
    from scireason.pipeline.task2_validation import prepare_task2_validation_bundle as _prepare_task2_validation_bundle

    def build_task2_validation_bundle(*args, **kwargs):
        bundle_dir = _prepare_task2_validation_bundle(*args, **kwargs)
        class _Result:
            def __init__(self, bundle_dir):
                self.bundle_dir = Path(bundle_dir)
                self.manifest_path = self.bundle_dir / 'task2_notebook_manifest.json'
        return _Result(bundle_dir)

    def load_task1_yaml(path):
        import yaml
        return yaml.safe_load(Path(path).read_text(encoding='utf-8'))

    def make_hvplot_payload(payload):
        import networkx as nx
        G = nx.DiGraph()
        for node in payload.get('nodes', []) or []:
            node_id = node.get('id') or node.get('term') or node.get('label')
            if node_id is not None:
                G.add_node(str(node_id), **dict(node))
        for edge in payload.get('edges', []) or []:
            src = edge.get('source')
            tgt = edge.get('target') or edge.get('object')
            if src is not None and tgt is not None:
                G.add_edge(str(src), str(tgt), **dict(edge))
        try:
            import hvplot.networkx as hvnx  # noqa: F401
            pos = nx.spring_layout(G, seed=7)
            plot = hvnx.draw(G, pos, with_labels=False, width=950, height=650)
            return G, plot
        except Exception:
            return G, None

print('Готово.')
print('Repo dir:', REPO_DIR)
print('Src dir:', SRC_DIR)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)


In [ ]:
# @title
# Вспомогательные функции (не нужно редактировать)
from pathlib import Path


def _save_uploaded_yaml(upload_value, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    if isinstance(upload_value, dict):
        name, meta = next(iter(upload_value.items()))
        content = meta['content']
        path = out_dir / name
        path.write_bytes(content)
        return path
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        name = meta.get('name') or 'task1.yaml'
        content = meta.get('content')
        path = out_dir / name
        path.write_bytes(bytes(content))
        return path
    raise ValueError('Не удалось прочитать загруженный файл. Загрузите YAML ещё раз.')


def _display_manifest(manifest_path: Path):
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    lines = [
        f"- **topic**: {manifest.get('topic')}",
        f"- **bundle_dir**: `{manifest.get('bundle_dir')}`",
        f"- **gold_graph**: `{manifest.get('gold_graph')}`",
        f"- **gold_triplets_csv**: `{manifest.get('gold_triplets_csv')}`",
    ]
    if manifest.get('auto_run_dir'):
        lines += [
            f"- **auto_run_dir**: `{manifest.get('auto_run_dir')}`",
            f"- **auto_graph_json**: `{manifest.get('auto_graph_json')}`",
            f"- **auto_triplets_csv**: `{manifest.get('auto_triplets_csv')}`",
        ]
    if manifest.get('comparison_summary'):
        lines.append(f"- **comparison_summary**: `{manifest.get('comparison_summary')}`")
    if manifest.get('reference_scout'):
        lines.append(f"- **reference_scout**: `{manifest.get('reference_scout')}`")
    display(Markdown('## Артефакты Task 2\n' + '\n'.join(lines)))
    return manifest


def _show_dataframe(csv_path: str, title: str, max_rows: int = 100):
    p = Path(csv_path)
    if not p.exists():
        display(Markdown(f'**{title}:** файл не найден'))
        return
    df = pd.read_csv(p)
    display(Markdown(f'## {title} ({len(df)} строк)'))
    display(df.head(max_rows))


def _show_graph(graph_json_path: str, html_path: str, title: str):
    display(Markdown(f'## {title}'))
    gj = Path(graph_json_path)
    hp = Path(html_path)
    if gj.exists():
        try:
            payload = json.loads(gj.read_text(encoding='utf-8'))
            try:
                _, plot = make_hvplot_payload(payload)
                if plot is not None:
                    display(plot)
            except Exception as e:
                display(Markdown(f'hvPlot не удалось построить: `{e}`'))
        except Exception as e:
            display(Markdown(f'Не удалось открыть JSON графа: `{e}`'))
    if hp.exists():
        try:
            display(HTML(hp.read_text(encoding='utf-8')))
        except Exception:
            display(Markdown(f'HTML-файл графа: `{hp}`'))


def _show_reference_scout(path: str):
    p = Path(path)
    if not p.exists():
        return
    scout = json.loads(p.read_text(encoding='utf-8'))
    rows = []
    if isinstance(scout, dict):
        for hit in scout.get('hits', []):
            for res in hit.get('results', []):
                rows.append({
                    'query': hit.get('query'),
                    'id': res.get('id'),
                    'title': res.get('title'),
                    'year': res.get('year'),
                    'url': res.get('url'),
                })
    elif isinstance(scout, list):
        for res in scout:
            if not isinstance(res, dict):
                continue
            rows.append({
                'query': ', '.join(res.get('trigger_queries') or []),
                'id': res.get('paper_id') or res.get('id'),
                'title': res.get('title'),
                'year': res.get('year'),
                'url': res.get('url'),
                'score': res.get('score'),
            })
    if rows:
        display(Markdown('## Reference scout'))
        display(pd.DataFrame(rows).head(100))


In [ ]:
# @title
# Форма Task 2 (запустите ячейку и заполните поля)

yaml_upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Загрузить YAML')
out_dir = W.Text(value='runs/task2_validation', description='Out dir', layout=W.Layout(width='100%'))
run_auto = W.Checkbox(value=True, description='Строить автоматический temporal KG')
reference_scout = W.Checkbox(value=True, description='Сгенерировать reference scout')
multimodal = W.Checkbox(value=True, description='Использовать multimodal pipeline')
run_btn = W.Button(description='Построить Task 2 bundle', button_style='success')
status = W.HTML()
out = W.Output()


def _on_run(_):
    with out:
        clear_output()
        try:
            status.value = '<b>Запуск...</b>'
            workdir = Path(tempfile.mkdtemp(prefix='task2_notebook_'))
            trajectory_path = _save_uploaded_yaml(yaml_upload.value, workdir)
            task1 = load_task1_yaml(trajectory_path)
            display(Markdown(f"# Загружен YAML\n- **topic**: {task1.get('topic')}\n- **submission_id**: `{task1.get('submission_id')}`"))

            bundle = build_task2_validation_bundle(
                trajectory_path,
                out_dir=Path(out_dir.value),
                include_auto_pipeline=bool(run_auto.value),
                multimodal=bool(multimodal.value),
                enable_reference_scout=bool(reference_scout.value),
            )
            manifest = _display_manifest(bundle.manifest_path)

            _show_dataframe(manifest['gold_triplets_csv'], 'Gold triplets')
            _show_graph(manifest['gold_graph'], manifest['gold_graph_html'], 'Gold graph')

            if manifest.get('auto_triplets_csv'):
                _show_dataframe(manifest['auto_triplets_csv'], 'Auto triplets')
            if manifest.get('auto_graph_json') and manifest.get('auto_graph_html'):
                _show_graph(manifest['auto_graph_json'], manifest['auto_graph_html'], 'Auto graph')
            if manifest.get('comparison_summary'):
                cmp = json.loads(Path(manifest['comparison_summary']).read_text(encoding='utf-8'))
                display(Markdown('## Comparison summary'))
                display(pd.DataFrame([cmp]))
            if manifest.get('reference_scout'):
                _show_reference_scout(manifest['reference_scout'])

            status.value = '<span style="color:green"><b>Готово.</b></span>'
        except Exception as e:
            status.value = f'<span style="color:red"><b>Ошибка:</b> {e}</span>'
            raise


run_btn.on_click(_on_run)

display(W.VBox([
    W.HTML('<h2>Task 2 — Validation UI</h2>'),
    yaml_upload,
    out_dir,
    run_auto,
    multimodal,
    reference_scout,
    run_btn,
    status,
    out,
]))


# CLI-режим (тот же workflow без notebook)

Если удобнее запускать всё одной командой из терминала, используйте:

```bash
top-papers-graph task2-bundle \
  --trajectory path/to/task1.yaml \
  --out-dir runs/task2_validation \
  --multimodal \
  --suggest-links
```

Эквивалентный основной алиас из репозитория:

```bash
top-papers-graph prepare-task2-validation \
  --trajectory path/to/task1.yaml \
  --out-dir runs/task2_validation \
  --multimodal \
  --suggest-links
```


# FAQ

## Что считается эталонным графом?
Эталонный граф строится **только из YAML Task 1** и полностью повторяет шаги эксперта, их evidence и связи между шагами.

## Что считается автоматическим графом?
Автоматический граф строится из **списка статей в YAML** и запускает встроенный конвейер репозитория: статьи → ingestion → temporal KG → review assets → report.

## Что делать, если автоматический граф почти пустой?
Это ожидаемо, если PDF недоступны, в статьях мало структурируемого текста или temporal evidence неявен. В этом случае ориентируйтесь на:
- `reference_scout.json`
- triplets CSV
- `comparison_summary.json`
- review templates из auto pipeline

## Как интерпретировать temporal поля?
- `start_date` / `end_date` — когда связь наблюдается / извлечена;
- `valid_from` / `valid_to` — когда открытие считается валидным;
- `temporal_basis` / `time_source` — откуда взялось время.
